#### RFM Classs code but in JP nb

In [ ]:


import matplotlib.pyplot as plt
import numpy
import pandas
import random
import sys

import seaborn
from datetime import datetime

# Set some options for printing all the columns
numpy.set_printoptions(precision = 10, threshold = sys.maxsize)
numpy.set_printoptions(linewidth = numpy.inf)

pandas.set_option('display.max_columns', None)
pandas.set_option('display.expand_frame_repr', False)
pandas.set_option('max_colwidth', None)

pandas.options.display.float_format = '{:,.10f}'.format


In [ ]:

transaction = pandas.read_csv(r'G:\My Drive\PROFESSIONAL\MAS24\Courses\CSP571\data\\rfm_Transactions.csv')


In [ ]:
# Calculate Number of days since 12/31/2020

t_date = pandas.to_datetime(transaction['Date'], format='%m/%d/%Y')
reference_date = datetime.strptime('12/31/2020', "%m/%d/%Y")
n_days = pandas.Series((t_date - reference_date) / numpy.timedelta64(1, 'D'), name = 'N Days')

In [ ]:


# Create the training data

train_data = transaction[['CustomerID', 'Date', 'Amount']].join(n_days)

# Define the aggregation procedure outside of the groupby operation
aggregations = {
    'N Days':'max',
    'CustomerID': 'count',
    'Amount': 'sum'
}

column_map = {'N Days': 'Recency', 'CustomerID': 'Frequency', 'Amount': 'Monetary'}

customer_data = train_data.groupby('CustomerID').agg(aggregations).rename(columns = column_map)
rfm_names = customer_data.columns

In [ ]:

# Determine the quintiles
quintile = customer_data.describe(percentiles = [0.2, 0.4, 0.6, 0.8])


In [ ]:


# Assign customers to groups
customer_group = pandas.DataFrame(numpy.where(numpy.isnan(customer_data),0,1), index = customer_data.index)

for q in ['20%','40%','60%','80%']:
   customer_group = customer_group + numpy.where(customer_data[rfm_names] > quintile.loc[q][rfm_names], 1, 0)

customer_group = customer_group.rename(columns = {0: 'Recency Score', 1: 'Frequency Score', 2: 'Monetary Score'})

In [ ]:



# Inspect bar charts of each group
for g in ['Recency Score', 'Frequency Score','Monetary Score']:
   group_prop = 100 * customer_group[g].value_counts(ascending = True, normalize = True)

   plt.figure(figsize = (12,4), dpi = 200)
   plt.bar(group_prop.index, group_prop, color = 'royalblue')
   plt.xlabel(g)
   plt.ylabel('Percentage of Customers')
   plt.xticks(range(1,6,1))
   plt.grid(axis = 'y', linestyle = '--')
   plt.margins(y = 0.1)
   plt.show()

# Merge the group assignments back to the customer data
customer_data = customer_data.join(customer_group)

customer_data['RFM Score'] = 100 * customer_data['Recency Score'] + 10 * customer_data['Frequency Score'] + customer_data['Monetary Score']

# Look at Monetary value by Recency and Frequency groups
xtab = pandas.crosstab(index = customer_data['Recency Score'], columns = customer_data['Frequency Score'],
                       values = customer_data['Monetary'], aggfunc = 'mean')
plt.figure(figsize = (10,8), dpi = 200)
seaborn.heatmap(xtab, cmap = 'Greens', cbar_kws={'label': 'Monetary Value'})
plt.gca().invert_yaxis()
plt.show()

# Copy the RFM Score back to the transaction data
transaction_rfm = transaction.merge(customer_data['RFM Score'], left_on = ['CustomerID'], right_on = customer_data.index)

# What kind of products do the customers with RFM score 555 buy?
focus_data = transaction_rfm[transaction_rfm['RFM Score'] == 555]

product_size = focus_data['ProductLine'].value_counts(ascending = True)

plt.figure(figsize = (10,6), dpi = 200)
plt.bar(product_size.index, product_size, color = 'royalblue')
plt.yticks(range(0,100,10))
plt.xlabel('Product Line')
plt.ylabel('Number of Transactions')
plt.grid(axis = 'y')
plt.show()


In [2]:
transaction = pandas.read_csv(r'G:\My Drive\PROFESSIONAL\MAS24\Courses\CSP571\data\\Online_Retail.csv')
transaction.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,12/1/2009 7:45,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,12/1/2009 7:45,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,12/1/2009 7:45,1.25,13085.0,United Kingdom


In [ ]:
# Calculate Number of days since 12/31/2020

t_date = pandas.to_datetime(transaction['Date'], format='%m/%d/%Y')
reference_date = datetime.strptime('12/31/2020', "%m/%d/%Y")
n_days = pandas.Series((t_date - reference_date) / numpy.timedelta64(1, 'D'), name = 'N Days')